# Cloud SmolVLA Loading Spike v0 (Generic VLA Backend)

Runs `vla_server.generic_vla_server` in this Colab runtime with `model_family="smolvla"`, to check whether a real SmolVLA/LeRobot checkpoint actually loads and what its raw `/predict` output looks like -- **not** whether the full local demo succeeds end to end. That's the local machine's job, unchanged.

**Role split** (see `docs/generic_vla_backend.md` / `docs/smolvla_cloud_loading_spike.md`):

```text
Local PC (unchanged): camera, PyBullet, SafetySupervisor, RobotBackend, RealVLAPolicyClient, episode recording
Colab (this notebook): only vla_server.generic_vla_server -- /health, /load_model, /predict. Never touches a robot.
```

**Success bar for this spike, in order** -- each is a strictly higher bar than the last, and stopping partway through is still useful information:

1. `generic_vla_server` imports and starts (`model_family="smolvla"`, no download at import time).
2. `/health` responds immediately.
3. `POST /load_model` either loads a real SmolVLA checkpoint (`model_status="loaded"`) **or** fails gracefully (`model_status="load_failed"` with a clear reason) -- either way, the server must still be alive afterward.
4. `POST /predict` with a dummy image returns either a normalized 7-DoF action (`SmolVLAActionAdapter` understood the raw output) or a structured error (it didn't -- see the raw output and fix the adapter, don't guess).

The full local demo (`run_full_recycling_cell_demo.py --policy-backend real-vla`) is exercised from your **local** machine in section 12 below, once this server is reachable -- and even that is expected to keep succeeding via fallback if SmolVLA doesn't load, exactly like every other Real VLA spike in this project.

## 1. Runtime / GPU check

SmolVLA is much lighter than OpenVLA-7B, but still benefits from a GPU. Reports what's actually available so you know what to expect from section 9's `/load_model` call before you get there.

In [ ]:
import platform

print(f"python_version: {platform.python_version()}")

try:
    import torch
    torch_installed = True
    torch_version = torch.__version__
    cuda_available = torch.cuda.is_available()
except ImportError:
    torch_installed = False
    torch_version = None
    cuda_available = False

print(f"torch_installed: {torch_installed}")
print(f"torch_version: {torch_version}")
print(f"cuda_available: {cuda_available}")

gpu_name = None
vram_total_gb = None
vram_free_gb = None
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    vram_total_gb = round(total_bytes / (1024 ** 3), 2)
    vram_free_gb = round(free_bytes / (1024 ** 3), 2)

print(f"gpu_name: {gpu_name}")
print(f"vram_total_gb: {vram_total_gb}")
print(f"vram_free_gb: {vram_free_gb}")

if not cuda_available:
    print("No CUDA GPU detected -- /load_model in section 9 will likely report model_status=load_failed "
          "with reason=no_cuda_gpu_available (or run on CPU if the loaded policy supports it). "
          "That is an environment limitation, not a bug -- sections 1-8 below are still worth completing.")

## 2. Clone the repo

Only `vla_server/`, `vla_adapters/`, and `policy/` (for `DummyOpenVLAPolicy`, used by `model_family="mock-action"` if you want to sanity-check the server itself first) are actually needed from this repo -- but cloning the whole thing is simplest. Replace `REPO_URL` with wherever you're hosting this repo.

In [ ]:
import os
import sys

REPO_URL = "REPLACE_WITH_YOUR_REPO_URL"  # TODO: replace with your GitHub repo URL, e.g. https://github.com/<you>/physical-ai-recycling-cell.git
REPO_DIR = "/content/physical-ai-recycling-cell"

if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !cd "$REPO_DIR" && git pull

sys.path.insert(0, REPO_DIR)

## 3. Install dependencies

Installed in a deliberately safe order: pip itself first, then LeRobot (which pulls in most of what SmolVLA needs), then anything LeRobot's install might have missed or under-pinned, then the plain server dependencies. If `pip install lerobot` fails outright, section 4's import-candidate probe still runs and tells you exactly what's missing -- this cell failing is not fatal to finding that out.

In [ ]:
%pip install -q -U pip
%pip install -q "lerobot" || echo "pip install lerobot failed -- see section 4 for what specifically is missing."
%pip install -q transformers accelerate safetensors huggingface_hub hf_transfer
%pip install -q pillow fastapi uvicorn requests nest_asyncio

## 4. Validate LeRobot/SmolVLA imports

LeRobot's package layout for policy classes has moved around across releases, so this tries several plausible import paths in turn (the same candidates `vla_server/model_loader.py` itself tries) and reports exactly which one worked, if any. If none work, this prints **"SmolVLA import path needs update"** -- that's your signal to check the installed `lerobot` version's actual module layout (`import lerobot; print(lerobot.__file__)` and browse from there) and add the correct path to `vla_server/model_loader.py`'s `_SMOLVLA_IMPORT_CANDIDATES`, not a sign anything else here is broken.

In [ ]:
import importlib

try:
    import lerobot
    print("lerobot:", getattr(lerobot, "__version__", "unknown version"), "at", lerobot.__file__)
except ImportError as exc:
    print("lerobot import FAILED:", exc)

_SMOLVLA_IMPORT_CANDIDATES = [
    "lerobot.common.policies.smolvla.modeling_smolvla:SmolVLAPolicy",
    "lerobot.policies.smolvla.modeling_smolvla:SmolVLAPolicy",
    "lerobot.common.policies.smolvla.smolvla:SmolVLAPolicy",
]

found = False
for candidate in _SMOLVLA_IMPORT_CANDIDATES:
    module_path, _, class_name = candidate.partition(":")
    try:
        module = importlib.import_module(module_path)
        policy_class = getattr(module, class_name)
        print(f"OK   {candidate} -> {policy_class}")
        found = True
        break
    except Exception as exc:
        print(f"FAIL {candidate}: {exc}")

if not found:
    print()
    print("SmolVLA import path needs update -- none of the candidates above matched this LeRobot install.")
    print("Falling back to a plain transformers.AutoModelForImageTextToText load is the next thing "
          "vla_server/model_loader.py tries (see section 9) -- it may still work even if the LeRobot-native path doesn't.")

## 5. Set model selection env vars

Reads `configs/vla_backend_smolvla_config.json`'s `model_id_or_path` as the default, but every value here is overridable without editing that file -- exactly the same env vars `vla_server/model_loader.py` and a real deployment would use:

```text
VLA_MODEL_FAMILY       smolvla
VLA_MODEL_ID_OR_PATH   HF Hub repo id or local path (overrides the config file if set)
VLA_LOCAL_FILES_ONLY   "1" to force local_files_only=True (skip network entirely)
VLA_DEVICE             cuda | cpu
VLA_DTYPE              bfloat16 | float16 | float32
```

In [ ]:
import json
import os

with open("/content/physical-ai-recycling-cell/configs/vla_backend_smolvla_config.json") as config_file:
    _default_config = json.load(config_file)

os.environ["VLA_MODEL_FAMILY"] = "smolvla"
os.environ["VLA_MODEL_ID_OR_PATH"] = _default_config.get("model_id_or_path", "lerobot/smolvla_base")  # override here if needed
os.environ["VLA_LOCAL_FILES_ONLY"] = "0"
os.environ["VLA_DEVICE"] = "cuda"  # falls back to cpu automatically inside model_loader.py if no GPU
os.environ["VLA_DTYPE"] = "bfloat16"  # try "float16" if your GPU doesn't support bfloat16 well
os.environ["VLA_BACKEND_CONFIG_PATH"] = "/content/physical-ai-recycling-cell/configs/vla_backend_smolvla_config.json"

for key in ["VLA_MODEL_FAMILY", "VLA_MODEL_ID_OR_PATH", "VLA_LOCAL_FILES_ONLY", "VLA_DEVICE", "VLA_DTYPE"]:
    print(f"{key}={os.environ[key]}")

## 6. Import the Generic VLA Backend app

This import is always fast (well under a second) regardless of `VLA_MODEL_FAMILY` -- it never downloads or loads a model. Model loading only happens later, explicitly, via `POST /load_model` (section 9).

In [ ]:
from vla_server.generic_vla_server import app, _MODEL_FAMILY, _ADAPTER

print(f"model_family: {_MODEL_FAMILY}")
print(f"adapter: {type(_ADAPTER).__name__}")

## 7. Start the FastAPI server

Runs uvicorn in a background thread so this cell returns and the notebook stays usable.

In [ ]:
import threading
import time

import nest_asyncio
import uvicorn

nest_asyncio.apply()

LOCAL_PORT = 9200

config = uvicorn.Config(app, host="127.0.0.1", port=LOCAL_PORT, log_level="info")
server = uvicorn.Server(config)

server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
time.sleep(2)
print(f"local server running on http://127.0.0.1:{LOCAL_PORT}")

## 8. Check /health

Expected right after section 7 (before `/load_model` in section 9):

```json
{
  "status": "ok",
  "model_family": "smolvla",
  "model_status": "not_loaded",
  "model_status_reason": "model load has not been requested",
  "model_id_or_path": "...",
  "local_files_only": false,
  "adapter": "SmolVLAActionAdapter",
  "version": "v0"
}
```

`model_id_or_path` reflects whatever section 5 set -- confirm it's what you expect before triggering an actual load in section 9.

In [ ]:
import requests

response = requests.get(f"http://127.0.0.1:{LOCAL_PORT}/health", timeout=10)
print(response.status_code)
print(response.json())

## 9. Trigger model loading

The only cell that ever attempts to actually load SmolVLA. Safe to re-run -- an in-progress or already-finished load is reported back as-is instead of restarting.

**How to read the result** (`model_status_reason` explains all of these):

| `model_status` | Meaning | Next step |
|---|---|---|
| `loaded` | Success | Continue to section 10 |
| `load_failed`, reason mentions `missing_dependency` / `SmolVLA import path needs update` | Section 3/4's install or import didn't work for this LeRobot version | Check section 4's output, fix the import candidate in `vla_server/model_loader.py` if needed |
| `load_failed`, reason mentions `repository not found` / `404` / similar | `VLA_MODEL_ID_OR_PATH` is wrong or private/gated | Fix the model id/path in section 5 |
| `load_failed`, reason mentions `CUDA out of memory` | Model+deps downloaded fine -- this Colab GPU tier doesn't have enough VRAM | Try a smaller model, `VLA_DTYPE=float16`, or a bigger GPU tier |
| `load_failed`, anything else | Recorded verbatim in `model_status_reason` | The server is still alive either way -- read the reason and adjust |

Every case above ends with **the server still running** -- `/health` and `/predict` both stay reachable, and `/predict` will respond a structured `model_not_loaded` error (triggering local fallback) instead of the whole server going down.

In [ ]:
response = requests.post(f"http://127.0.0.1:{LOCAL_PORT}/load_model", timeout=600)
print(response.status_code)
print(response.json())

print()
health = requests.get(f"http://127.0.0.1:{LOCAL_PORT}/health", timeout=10).json()
print("model_status after load attempt:", health["model_status"])
print("model_status_reason:", health["model_status_reason"])

## 10. /predict with a dummy image

Goal here is to see the raw output shape, not to prove full-demo correctness. If `model_status != "loaded"`, this will return the same structured `model_not_loaded` error as `/health` already showed -- still useful to confirm end to end.

If it *is* loaded: `SmolVLAActionAdapter` (`vla_adapters/smolvla_adapter.py`) tries several raw-output shapes (plain 7-vector, `{"action": ...}`, chunked `{"actions": [...]}`, batched `[B, T, 7]`, numpy/torch tensors) before giving up -- if it can't interpret this model's actual raw output, the response's `info.raw_output_summary` shows the shape/type it saw (never the full tensor) and `info.reason` explains why, which is exactly what to use to extend the adapter's `_peel_to_vector()` for this model.

In [ ]:
import base64
import io

import numpy as np
from PIL import Image

dummy_image = np.zeros((240, 320, 3), dtype=np.uint8)
dummy_image[100:140, 140:180] = [80, 140, 220]

buffer = io.BytesIO()
Image.fromarray(dummy_image).save(buffer, format="JPEG", quality=90)
image_payload = {
    "encoding": "jpg_base64",
    "shape": list(dummy_image.shape),
    "data": base64.b64encode(buffer.getvalue()).decode("ascii"),
}

payload = {
    "instruction": "플라스틱 병을 플라스틱 수거함에 넣어줘",
    "robot_state": {"end_effector_position": [0.5, 0.0, 0.5], "held_object": False, "task_status": "running"},
    "task_goal": {"action": "pick_and_place", "target_object": "plastic_bottle", "target_bin": "plastic_bin"},
    "target_object_position": [0.4, -0.1, 0.05],
    "bin_position": [0.3, 0.35, 0.05],
    "step_index": 0,
    "phase": "move_to_object",
    "image": image_payload,
}

response = requests.post(f"http://127.0.0.1:{LOCAL_PORT}/predict", json=payload, timeout=60)
print(response.status_code)
data = response.json()
print(json.dumps(data, indent=2, ensure_ascii=False)[:3000])

if response.status_code == 200 and data.get("action") is not None:
    print()
    print(f"PASS -- action_len={len(data['action'])}")
elif isinstance(data.get("detail"), dict) and data["detail"].get("raw_output_summary"):
    print()
    print("Adapter could not interpret the raw output -- see raw_output_summary/reason above.")
    print("This is exactly what tells you what to add to SmolVLAActionAdapter._peel_to_vector().")
else:
    print()
    print("model_not_loaded -- see section 9's result for why.")

## 11. Public tunnel (cloudflared)

cloudflared's quick tunnel needs no account/token, unlike ngrok -- the default recommendation here.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("Run the next line, then watch the output for a https://<random>.trycloudflare.com URL:")
print(f"!./cloudflared tunnel --url http://127.0.0.1:{LOCAL_PORT}")

In [ ]:
# Run this in its own cell (it blocks/streams output) -- copy the
# printed https://<random>.trycloudflare.com URL once it appears.
!./cloudflared tunnel --url http://127.0.0.1:{LOCAL_PORT}

## 12. From your local machine

```bash
python scripts/update_colab_vla_config.py \
  --base-url https://새로운주소.trycloudflare.com \
  --config configs/vla_backend_smolvla_config.json

python -m benchmark.probe_generic_vla_server \
  --vla-config configs/vla_backend_smolvla_config.json \
  --with-image

python -m benchmark.run_full_recycling_cell_demo \
  --policy dummy-openvla \
  --policy-backend real-vla \
  --real-vla-config configs/vla_backend_smolvla_config.json \
  --real-vla-fallback-backend local-dummy \
  --image-path data/test_images/recyclable_scene.jpg \
  --headless
```

All three are expected to succeed (`PASS`/`PASS_WITH_FALLBACK`/`final_status: success`) whether or not SmolVLA actually loaded in this Colab session -- `--real-vla-fallback-backend local-dummy` is what makes that true. See `docs/smolvla_cloud_loading_spike.md` for the full local walkthrough.

## 13. Session notes and interpretation summary

- **This tunnel/session is temporary**, same as the OpenVLA spike (`docs/colab_vla_server_spike.md`) -- it ends on disconnect/idle timeout/tab close, and a new session gets a new URL (re-run `scripts/update_colab_vla_config.py` locally each time).
- **`notebooks/colab_vla_server_spike_v0.ipynb` (OpenVLA-specific) is untouched** and remains available as an optional/deprecated path for OpenVLA-specific experiments (Google Drive cache, `openvla-dryrun` mode). This notebook (SmolVLA via the Generic VLA Backend) is the recommended default going forward.
- **Nothing here ever executes on a robot.** This server only ever proposes actions (or refuses to); the local `RealVLAPolicyClient` -> `policy/vla_action_postprocessor.py` -> `SafetyGate`/`SafetySupervisor` -> `RobotBackend` chain decides what actually happens, unchanged from every other Real VLA backend in this project.

| Stage reached | What it means |
|---|---|
| Section 8 `/health` responds | Tunnel/HTTP plumbing works -- true regardless of model loading |
| Section 9 `model_status=loaded` | SmolVLA (or your configured checkpoint) actually loaded on this GPU |
| Section 9 `model_status=load_failed` | Environment limitation (dependency/model id/VRAM) -- recorded, not a crash; see the table in section 9 |
| Section 10 returns `action` (200) | `SmolVLAActionAdapter` understood this model's raw output shape |
| Section 10 returns structured error with `raw_output_summary` | Adapter needs a new shape case -- extend `_peel_to_vector()` using that summary |
| Section 12's local commands all succeed | The whole pipeline (Colab server, tunnel, local client, fallback) works, independent of whether SmolVLA itself loaded |